In [ ]:
from varphot import nbui as ui

ui.show(ui.header("03", "Bericht — AAVSO-Export prüfen und schreiben",
    "AAVSO-Datei (Extended Format) in `PHOTOMETRIE_PY_DATA/exports/`, geprüft auf Format und Inhalt. Einreichen: WebObs → „File Upload“ (kein Automatismus).",
    flow=[("Schritt 1 — Formular starten", "Die Zellen von oben nach unten ausführen (▶ oder Shift+Enter), bis das **Formular mit den Eingabefeldern** erscheint"),
          ("Schritt 2 — Prüfen", "Meist Defaults übernehmen; der **Hinweiskasten im Formular** zeigt: **grün** = ok · **orange** = prüfen · **rot** = Fehler"),
          ("Schritt 3 — Automatisch", "Weiter mit dem Abschnitt **„Ab hier automatisch“**: bei der nächsten Code-Zelle den Pfeil (▶) öffnen → **„Diese Zelle und alle folgenden“** ausführen")],
    method=ui.current_method(), fall_title="Meldefilter nach Fall", texts={"mono": "`V`/`B`/`R`, untransformiert oder nach NB08 transformiert", "monorgb": "`TG`/`TB`/`TR` (`TRANS=NO`) oder nach NB08 `V`/`B`/`R` (`TRANS=YES`)",
           "osc": "`TG`/`TB`/`TR` (`TRANS=NO`) oder nach NB08 `V`/`B`/`R` (`TRANS=YES`)", "dslr": "`TG`/`TB`/`TR` (`TRANS=NO`) oder nach NB08 `V`/`B`/`R` (`TRANS=YES`)"}))

In [ ]:
from varphot import nbui as ui

ui.show(ui.step("03", None, "Felder im Überblick", blocks=[ui.fields([
    ("Eigene Beobachtungen", "**kein Häkchen** (Testdaten → Code `TST01`)", "**Häkchen setzen**: erst dann geht Ihr AAVSO-Code in die Datei"),
    ("Betriebsart", "*untransformiert* lassen", "Fall A: *untransformiert* oder mit NB08 *transformiert*; Fälle B–D: *untransformiert* → `TG`/`TB`/`TR`"),
    ("OBSTYPE", "*laut Plan* lassen", "Mono: `CCD`, Farbe: `DSLR` (bei Unsicherheit mit AAVSO klären)"),
    ("KI-Zweitlesung", "Häkchen lassen", "Häkchen lassen (lokal, bis ~2 min)")], ("Testlauf (LCO)", "Eigene Aufnahmen"))]))

In [ ]:
from varphot import nbui as ui
import os, json, datetime
from pathlib import Path
import numpy as np
import pandas as pd
import ipywidgets as widgets
from IPython.display import display

from varphot import (Session, TransformSolution, build_extended, notes_string, observations_from_lightcurve, validate_extended, load_plan, __version__)
from varphot.observer import data_dir, load_config
from varphot import workflow as W
from varphot.hints import bind, nb03_hints

SESS = Session.current()
PLAN = load_plan(SESS.plan) if SESS.plan and Path(SESS.plan).exists() else None
RESULTS = SESS.results_dir()
EXPORTS = (data_dir() if data_dir().is_dir() else SESS.path()) / "exports"
LC = pd.read_csv(RESULTS / "zeitreihe.csv")
LC_TR = pd.read_csv(RESULTS / "zeitreihe_transformiert.csv") if (RESULTS / "zeitreihe_transformiert.csv").exists() else None
FC = json.loads(Path(SESS.field_config).read_text()) if SESS.field_config and Path(SESS.field_config).exists() else None
OBS, _own = load_config()
ui.say(f"Sitzung {SESS.workspace}\n  {SESS.target} · {len(LC)} Punkte" + (f" · transformiert {len(LC_TR)}" if LC_TR is not None else "") + f" · Beobachtercode {OBS.aavso_code or '— fehlt (NB00 Schritt 0b) —'}")

_st, _lay = {"description_width": "200px"}, widgets.Layout(width="680px")
w_real = widgets.Checkbox(description="Das sind meine eigenen Beobachtungen (nicht die Testdaten)", value=SESS.source != "lco-demo", indent=False, layout=_lay)
w_mode = widgets.Dropdown(description="Betriebsart:", options=[("untransformiert (V bei Mono, TG/TB/TR bei Farbe)", "untransformiert"), ("transformiert (V/B/R mit TRANS=YES, braucht NB08)", "transformiert")], value=os.environ.get("VARPHOT_NB06_MODE", "untransformiert"), style=_st, layout=_lay)
w_band = widgets.Dropdown(description="Band (transformiert):", options=["V", "B", "R"], value=os.environ.get("VARPHOT_NB06_BAND", "V"), style=_st, layout=_lay)
w_obstype = widgets.Dropdown(description="OBSTYPE:", options=[("laut Plan (Mono: CCD, Farbe: DSLR)", ""), ("CCD", "CCD"), ("DSLR", "DSLR")], value=os.environ.get("VARPHOT_OBSTYPE", ""), style=_st, layout=_lay)
w_obst_ok = widgets.Checkbox(description="OBSTYPE bei der AAVSO geklärt (Hinweis ausblenden)", value=bool(os.environ.get("VARPHOT_OBSTYPE_CONFIRMED")), indent=False, layout=_lay)
w_chart = widgets.Text(description="CHART (Sequenz-ID):", value=os.environ.get("VARPHOT_CHART_ID", ""), placeholder="leer = aus NB01 (VSP)", style=_st, layout=_lay)
w_range = widgets.Text(description="Erwarteter Helligkeitsbereich:", value=os.environ.get("VARPHOT_EXPECTED_RANGE", ""), placeholder='z. B. 3.3,4.5 (leer = grob aus dem Plan)', style=_st, layout=_lay)
w_tf = widgets.Text(description="Transformationsdatei (NB08):", value=os.environ.get("VARPHOT_TRANSFORM_FILE", ""), placeholder="nur bei transformiert", style=_st, layout=_lay)
from varphot.aireview import detect_cloud_backends, load_ai_config
_cl = {b.kind: b for b in detect_cloud_backends(load_ai_config())}
w_ai = widgets.Dropdown(description="KI-Zweitlesung:", style=_st, layout=_lay, value="aus" if os.environ.get("VARPHOT_AI_REVIEW", "1") == "0" else "lokal",
                        options=[("Aus", "aus"), ("Lokal (Heimnetz, bis ca. 2 Minuten)", "lokal"), ("Claude (Cloud)" + ("" if _cl["claude-cli"].usable else ": nicht verfügbar"), "claude"),
                                 ("Gemini (Cloud): noch nicht eingerichtet", "gemini")])
w_hint = widgets.HTML()
bind(w_hint, [w_real, w_mode, w_band, w_tf, w_ai],
     lambda: nb03_hints(w_real.value, OBS.aavso_code, w_mode.value, w_band.value, w_tf.value, w_ai.value, lco=SESS.source == "lco-demo"))
display(widgets.VBox([w_hint, w_real, w_mode, w_band, w_obstype, w_obst_ok, w_chart, w_range, w_tf, w_ai]))

## Ab hier automatisch

Bei der **nächsten Code-Zelle** den Pfeil (▶) öffnen → **„Diese Zelle und alle folgenden“** ausführen.

In [ ]:
SOL = TransformSolution.load(w_tf.value.strip()) if w_tf.value.strip() else None
TAG = SESS.tag or RESULTS.name
ARCH = W.archive_reason(SESS)
IS_TESTDATA = bool(ARCH) or not w_real.value
if ARCH:
    ui.say("⚠ Fremde Archivdaten erkannt:", ARCH)
OBSCODE = "TST01" if IS_TESTDATA else (OBS.aavso_code or "")
STAR = (os.environ.get("VARPHOT_STAR") or (PLAN.target if PLAN else (FC["ziel"]["name"].replace("* ", "") if FC else SESS.target))).upper()
if IS_TESTDATA:
    ui.say("⚠ TESTDATEN (nicht deine Beobachtung): Code TST01, Dateiname markiert — nicht einreichen." + ("" if ARCH else " Eigene Daten: oben das Häkchen setzen."))

In [ ]:
MODE_T = w_mode.value == "transformiert"
OBSTYPE = w_obstype.value or (PLAN.obstype if PLAN else "CCD")
COLOUR = bool(PLAN and PLAN.method in ("osc", "dslr", "monorgb"))
OBSTYPE_UNCONFIRMED = COLOUR and not w_obst_ok.value
if OBSTYPE_UNCONFIRMED:
    ui.say("ENTSCHEIDUNG nötig — OBSTYPE bei Farbkamera/RGB-Filter: " + OBSTYPE + " (Vorgabe aus dem Plan).\n"
          "  Lesart A (AAVSO-Forum, Hawkins): 'DSLR' gilt für DSLR, Farb-CMOS (OSC) und Mono-CMOS mit RGB-Filtern.\n"
          "  Lesart B (Format-Seite): 'CMOS → CCD'.\n"
          "  Ändern: oben OBSTYPE wählen; nach Klärung bei der AAVSO das Häkchen setzen (dann entfällt dieser Hinweis).")
if MODE_T:
    if LC_TR is None or SOL is None:
        raise RuntimeError("transformierter Export braucht zeitreihe_transformiert.csv (NB02, Abschnitt Transformation) und die Transformationsdatei aus NB08 (Formular oben)")
    BAND = w_band.value
    ok_band = SOL.validation.get(BAND, {})
    if not ok_band or ok_band.get("rms_transformed", 1) > ok_band.get("rms_standardized", 0) or ok_band.get("rms_transformed", 1) > 0.06:
        raise RuntimeError(f"Band {BAND}: Gütetest in NB08 nicht bestanden (Transformation verbessert nicht oder rms > 0,06 mag) — als TG/TB/TR untransformiert melden")
    FILT, TRANS = BAND, True
    DATA = LC_TR.rename(columns={BAND: "mag", "V_err": "mag_err"})[["jd_mid", "mag", "mag_err", "airmass"] + (["V_check"] if (BAND == "V" and "V_check" in LC_TR) else [])]
    if "V_check" in DATA:
        DATA = DATA.rename(columns={"V_check": "check_mag"})                  # transformierte Check-Magnitude (Handbuch §6.7)
    COEF = dict(Tv_bv=round(SOL.Tv_bv, 4) if BAND == "V" else None, Tbv=round(SOL.Tbv, 4))
else:
    FILT = PLAN.aavso_filter if PLAN else "V"
    TRANS = False
    DATA = LC
    COEF = {}
ui.say(f"Meldung: Filter {FILT}, TRANS={'YES' if TRANS else 'NO'}, OBSTYPE {OBSTYPE}, {len(DATA)} Punkte")

In [ ]:
CHART_ID = w_chart.value.strip() or ((FC or {}).get("vsp", {}).get("chartid") or "")   # Sequenz-ID: aus NB01 (VSP-Antwort), sonst von Hand
comps = [s for s in (FC["sterne"] if FC else []) if s["rolle"] == "comp"]
check = next((s for s in (FC["sterne"] if FC else []) if s["rolle"] == "check"), None)
n_gaia = sum(str(s["auid"]).startswith("Gaia") for s in comps)
if not comps:
    CHART, WARN_CH = "na", "keine field_config.json: Comps unbekannt"
elif n_gaia:
    CHART, WARN_CH = "Gaia DR3 V-deriv", f"{n_gaia} von {len(comps)} Comps Gaia-abgeleitet → keine AAVSO-Sequenz-ID; als Notbehelf kennzeichnen"
elif CHART_ID:
    CHART, WARN_CH = CHART_ID, ""
else:
    CHART, WARN_CH = "na", "CHART_ID fehlt: field_config.json ohne Sequenz-ID — NB01 neu ausführen oder oben eintragen"
CHECK_LABEL = str(check["label"]) if check else None
KNAME = CHECK_LABEL if (not MODE_T or "check_mag" in DATA) else None
ui.say(f"Comps: {len(comps)} (davon Gaia-abgeleitet: {n_gaia}), Check-Stern: {KNAME or '—'}, CHART: {CHART}")
if WARN_CH:
    ui.say("⚠", WARN_CH)

In [ ]:
from varphot import nbui as ui

ui.show(ui.step("03", None, "Datei erzeugen und prüfen", blocks=[ui.kv([
    ("Format", "Pflichtparameter, 15 Felder, Längen, Filtercodes, Ensemble-Regel, JD"),
    ("Inhalt", "`check_plausibility`: Zeit, doppelte Zeilen, Helligkeitsbereich (auch gegen VSX), Fehler, Filter/`TRANS`/`OBSTYPE`, `CHART`, Check-Stern, Sprünge, Sternhöhe und Luftmasse aus Ort und Zeit"),
    ("Folgen", "`ERROR` verhindert das Schreiben, `WARN` steht als „Offene Punkte“ im Bericht"),
    ("Reifegrad", "was ist nicht an echten Daten geprüft")])]))

In [ ]:
core_note = "Ziel gesaettigt, Kernmaske-Korrektur" if ("core_corrected" in LC and bool(LC["core_corrected"].any()) and not MODE_T) else ""
NOTES = notes_string("varphot %s; Ensemble %d Comps%s%s" % (__version__, int(np.nanmedian(LC["n_comps"])) if "n_comps" in LC else 0,
                     "; " + core_note if core_note else "", "; Comps teils Gaia-abgeleitet (K)" if n_gaia else ""), **COEF)
OBSV = observations_from_lightcurve(DATA, STAR, FILT, TRANS, chart=CHART, kname=KNAME, kmag_col=("check_mag" if (KNAME and "check_mag" in DATA) else None),
                                    notes=NOTES)
TEXT = build_extended(OBSV, OBSCODE or "XXXXX", f"varphot {__version__}", OBSTYPE, comments=[f"Quelle: {TAG}", "Zeit: JD (UT) Mitte der Belichtung; BJD_TDB steht in zeitreihe.csv"] + (["TESTDATEN - NICHT EINREICHEN"] if IS_TESTDATA else []))
ISSUES = validate_extended(TEXT)
if not OBSCODE:
    ISSUES.append("Beobachtercode fehlt (NB01 Abschnitt 0 / AAVSO-Konto)")
ui.say(f"{len(OBSV)} Beobachtungen; Formatprüfung: " + ("✓ keine Probleme" if not ISSUES else f"{len(ISSUES)} Problem(e)"))
for i in ISSUES[:15]:
    ui.say("  ✗", i)
ui.say("\n".join(TEXT.splitlines()[:12]))

In [ ]:
from varphot import check_plausibility, maturity_warnings
if w_range.value.strip():
    EXPECTED = tuple(float(x) for x in w_range.value.split(","))
elif PLAN and PLAN.v_max is not None:
    EXPECTED = (PLAN.v_max - 0.5, PLAN.v_max + 2.0)                    # grob (Amplituden bis ~2 mag); genauer mit VARPHOT_EXPECTED_RANGE
else:
    EXPECTED = None
_ref = {"V": "V", "TG": "V", "B": "B", "TB": "B", "R": "R", "TR": "R"}.get(FILT)
KEXP = float(check[_ref]) if (check and _ref and check.get(_ref) is not None and check[_ref] == check[_ref]) else None
# Katalogbereich (VSX) und Ort/Zeit (KONZEPT §11.16): nur Warnungen; ohne Netz oder ohne Standort entfällt die Prüfung mit einem Hinweis, nie mit einem Abbruch
from varphot import vsx
from varphot.observer import all_sites
_z = (FC or {}).get("ziel", {})
CATALOG, _vsx_msg = (None, "Katalogbereich nicht geprüft: Bereich wurde oben von Hand angegeben") if w_range.value.strip() else vsx.lookup_range(_z.get("name") or SESS.target, ra=_z.get("ra", getattr(PLAN, "ra_deg", None)), dec=_z.get("dec", getattr(PLAN, "dec_deg", None)), band=_ref or "V")
_site_name = (getattr(PLAN, "site_name", "") or OBS.site_name) if SESS.source != "lco-demo" else "LCO McDonald (Testdaten delta Cep)"
SITE = next((s for s in all_sites(_own) if s.name == _site_name), None)
_radec = (_z["ra"], _z["dec"]) if "ra" in _z else ((PLAN.ra_deg, PLAN.dec_deg) if PLAN else None)
FINDINGS = check_plausibility(OBSV, OBSTYPE, EXPECTED, kmag_expected=KEXP, catalog=CATALOG, site=SITE, target_radec=_radec)
if OBSTYPE_UNCONFIRMED:
    from varphot import Finding
    FINDINGS.append(Finding("WARN", "obstype_unbestaetigt", f"OBSTYPE={OBSTYPE} bei Farbkamera/RGB-Filter noch nicht mit der AAVSO geklärt (oben das Häkchen setzen, sobald geklärt)"))
_keys = ["aavso_format", "lightcurve_mono"] + (["transform_fit", "transform_lightcurve"] if MODE_T else []) + (["monorgb"] if PLAN and PLAN.method == "monorgb" else []) \
        + (["cfa_split"] if PLAN and PLAN.method in ("osc", "dslr") else []) + (["obstype_colour"] if COLOUR else [])
MATURE = maturity_warnings(_keys)
ERRORS = [f for f in FINDINGS if f.level == "ERROR"]
WARNS = [f for f in FINDINGS if f.level == "WARN"]
ui.say(f"Inhaltsprüfung: {len(ERRORS)} Fehler, {len(WARNS)} Warnung(en); Reifegrad-Hinweise: {len(MATURE)}")
if CATALOG:
    ui.say(f"  Katalogbereich: {CATALOG.source} {CATALOG.lo:g}–{CATALOG.hi:g} {CATALOG.band} ({CATALOG.name})")
elif _vsx_msg:
    ui.say("  ○", _vsx_msg)
ui.say("  " + (f"Ort/Zeit geprüft für {SITE.name}" if SITE and _radec else "○ Ort/Zeit nicht geprüft: Standort oder Zielkoordinaten fehlen (NB00 Schritt 0b)"))
for f in ERRORS + WARNS:
    ui.say(" ", "✗" if f.level == "ERROR" else "⚠", f.code + ":", f.text)
for m in MATURE:
    ui.say("  ○", m)

In [ ]:
from varphot import nbui as ui

ui.show(ui.step("03", None, "Speichern und Zusammenfassung",
    intro="Die Datei entsteht nur, wenn Format **und** Inhaltsprüfung ohne Fehler sind. Einreichen: WebObs → „File Upload“, Datei wählen (kein Automatismus)."))

In [ ]:
today = datetime.date.today().isoformat()
name = f"aavso_{STAR.replace(' ', '_')}_{FILT}_{today}" + ("_TESTDATEN_NICHT_EINREICHEN" if IS_TESTDATA else "") + ".txt"
if ISSUES or ERRORS:
    ui.say("NICHT gespeichert — erst die Probleme beheben (Format und Inhaltsprüfung).")
else:
    EXPORTS.mkdir(parents=True, exist_ok=True)
    (EXPORTS / name).write_text(TEXT, encoding="utf-8")
    ui.say("gespeichert:", EXPORTS / name)
    sm = pd.Series(dict(Stern=STAR, Filter=FILT, TRANS="YES" if TRANS else "NO", OBSTYPE=OBSTYPE, Punkte=len(OBSV), Zeitraum=f"JD {DATA['jd_mid'].min():.3f} – {DATA['jd_mid'].max():.3f}",
                        mittlerer_Fehler_mag=round(float(DATA["mag_err"].median()), 3), Comps=len(comps), Gaia_Comps=n_gaia, CHART=CHART, Testdaten=IS_TESTDATA))
    ui.show(ui.table(sm))
    (EXPORTS / (name[:-4] + "_bericht.md")).write_text("# Export " + name + "\n\n" + "\n".join(f"- **{k}**: {v}" for k, v in sm.items()) + "\n\nWarnungen: " + (WARN_CH or "keine") + "\n\n## Offene Punkte (vor dem Einreichen entscheiden)\n" + ("\n".join(f"- {f}" for f in WARNS) or "- keine Warnungen der Inhaltsprüfung") + "\n\n## Reifegrad (nicht an echten Daten geprüft)\n" + ("\n".join(f"- {m}" for m in MATURE) or "- alle benutzten Funktionen an echten Daten geprüft") + "\n", encoding="utf-8")

In [ ]:
from varphot import nbui as ui

ui.show(ui.step("03", None, "KI-Zweitlesung", tag="optional",
    intro="Eine **lokale** KI liest den Berichtstext gegen eine feste Prüfliste. Nur Anmerkungen, sie ändert und rechnet nichts nach. Der Text verlässt den Rechner und das Heimnetz nicht.",
    blocks=[ui.note("warn", "Kleine Modelle haben Fehlalarme und übersehen Fehler (KONZEPT §11.14).")]))

In [ ]:
from varphot import detect_backends, choose_backend, ai_review
from varphot.aireview import ANON_NOTE, SYSTEM_PROMPT, detect_cloud_backends, load_ai_config
from varphot.anonymize import AnonymizationError, collect_secrets, prepare_for_cloud, restore, save_sent_text, summary
AI_RESULT = None
_mode = w_ai.value
if _mode == "aus" or ISSUES or ERRORS:
    ui.say("KI-Zweitlesung übersprungen" + (" (Datei nicht geschrieben)" if (ISSUES or ERRORS) else " (abgeschaltet)"))
else:
    _bfile = EXPORTS / (name[:-4] + "_bericht.md")
    _msg = (_bfile.read_text(encoding="utf-8") + "\n\n## Kopf der Datei\n" + "\n".join(TEXT.splitlines()[:12])
            + "\n\n## Ergebnis der festen Prüfungen\n" + ("\n".join(str(f) for f in FINDINGS) or "(keine)"))
    def _append(res, label):
        with open(_bfile, "a", encoding="utf-8") as fh:
            fh.write(f"\n## KI-Zweitlesung (unverbindlich; {label}, {res.seconds:g} s)\n\n" + res.text + "\n")
    if _mode == "lokal":
        _bk = choose_backend(detect_backends())
        if _bk is None:
            ui.say("KI-Zweitlesung: kein Weg verfügbar (optional) — siehe Vorbedingungen in NB00")
        else:
            ui.say(f"KI-Zweitlesung mit {_bk.label()} … (dauert je nach Modell bis ca. 2 Minuten)")
            AI_RESULT = ai_review(_msg, _bk, timeout=300)
            if AI_RESULT.error:
                ui.say("KI-Zweitlesung fehlgeschlagen:", AI_RESULT.error, "— der Bericht bleibt ohne sie gültig")
            else:
                ui.say(AI_RESULT.text)
                _append(AI_RESULT, AI_RESULT.backend)
    elif _mode == "gemini":
        ui.show(ui.note("info", "**Gemini ist noch nicht eingerichtet.** Der Bericht bleibt ohne KI-Zweitlesung gültig; „Lokal“ oder „Claude“ wählen."))
    else:
        _cfg = load_ai_config()
        _bk = next(b for b in detect_cloud_backends(_cfg) if b.kind == "claude-cli")
        if not _bk.usable:
            ui.show(ui.result("warn", "Claude ist nicht verfügbar", "", [ui.body(ui.para(_bk.detail))]))
        else:
            try:
                _anon = prepare_for_cloud(_msg, collect_secrets(OBS, _own, _cfg), max_chars=_cfg["cloud"]["max_chars"])
            except AnonymizationError as e:
                ui.show(ui.result("error", "Nicht gesendet", "Anonymisierung unvollständig", [ui.body(ui.para(str(e) + " Der Bericht bleibt ohne KI-Zweitlesung gültig."))]))
            else:
                ui.show(ui.cloud_preview("Claude", summary(_anon), _anon.text))
                _w_ok = widgets.Checkbox(description="Ich habe die Vorschau gelesen und bestätige, dass dieser Text an Claude gesendet wird.", value=False, indent=False, layout=widgets.Layout(width="760px"))
                _w_go = widgets.Button(description="Jetzt an Claude senden", button_style="warning", icon="paper-plane", layout=widgets.Layout(width="260px"))
                _w_out = widgets.Output()
                def _send(_b):
                    global AI_RESULT
                    with _w_out:
                        _w_out.clear_output()
                        if not _w_ok.value:
                            ui.say("⚠ Bitte zuerst die Bestätigung ankreuzen. Es wurde nichts gesendet.")
                            return
                        _w_go.disabled = _w_ok.disabled = True
                        ui.say("Sende an Claude … (dauert bis ca. 2 Minuten)")
                        _res = ai_review(_anon.text, _bk, system=SYSTEM_PROMPT + ANON_NOTE, timeout=300)
                        if _res.error:
                            ui.show(ui.result("error", "KI-Zweitlesung fehlgeschlagen", "", [ui.body(ui.para(_res.error + " Der Bericht bleibt ohne sie gültig."))]))
                            return
                        _res.text = restore(_res.text, _anon.mapping)
                        _sent = save_sent_text(EXPORTS, name[:-4], _anon, "Claude", _bk.model or "")
                        _append(_res, "Claude (Cloud)")
                        AI_RESULT = _res
                        ui.show(ui.ai_answer("Claude", _res.seconds, _res.text, f"Die Platzhalter wurden lokal zurückübersetzt. Gesendeter Text: `{_sent.name}`"))
                _w_go.on_click(_send)
                display(widgets.VBox([_w_ok, _w_go, _w_out]))